# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [6]:
# Updated installation cell
!pip install -U langchain langchain-openai langchain-classic langchain-community pandas

In [13]:
import pandas as pd
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import LLMChain

from google.colab import userdata
OPENAI_API_KEY  = userdata.get('OPENAI_API_KEY')
HF_TOKEN = userdata.get('HF_TOKEN')

In [2]:
#!pip install pandas

In [14]:
import pandas as pd
df = pd.read_csv('./Data.csv')

In [15]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\n,I loved this product. But they only seem to l...


## LLMChain

In [16]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import LLMChain

In [18]:
#Replace None by your own value and justify
llm = ChatOpenAI(temperature=0.7, api_key=OPENAI_API_KEY)


In [34]:
prompt = ChatPromptTemplate.from_template( #Write a query that would take a variable to describe any product
  """Does {product} has a good review?
  """
)

In [35]:
chain = LLMChain(llm=llm, prompt=prompt)

In [36]:
product = "Waterproof Phone Pouch"
chain.invoke(product)

{'product': 'Waterproof Phone Pouch',
 'text': 'It depends on the specific brand and model of the waterproof phone pouch. Generally, waterproof phone pouches have good reviews as they are designed to protect phones from water damage while allowing users to still use their phones. It is recommended to read customer reviews and ratings before purchasing a waterproof phone pouch to ensure that it meets your needs and expectations.'}

## SimpleSequentialChain

In [23]:
from langchain_classic.chains import SimpleSequentialChain

In [44]:
llm = ChatOpenAI(temperature=0.9, api_key=OPENAI_API_KEY)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    """This is a list of products stored in a pandas dataframe
    Input: {dataframe}

    Return me the list of product along with reviews in a CSV format which has positive reviews.
  """
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)

In [43]:

# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    """Out of all the product names and corresponding reviews {input},
      return me the product name which has the best review.
    """
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [45]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [46]:
overall_simple_chain.run(product)



> Entering new SimpleSequentialChain chain...
Product,Review
Waterproof Phone Pouch,"Great product! It kept my phone completely dry during my beach vacation."
Waterproof Phone Pouch,"Highly recommend this waterproof phone pouch. I used it while kayaking and it worked perfectly."
Waterproof Phone Pouch,"The waterproof phone pouch was a lifesaver during our snorkeling trip. No water damage to my phone at all."
The Waterproof Phone Pouch with the review "Great product! It kept my phone completely dry during my beach vacation."

> Finished chain.


'The Waterproof Phone Pouch with the review "Great product! It kept my phone completely dry during my beach vacation."'

**Repeat the above twice for different products**

## SequentialChain

In [48]:
from langchain_classic.chains import SequentialChain

In [62]:
llm = ChatOpenAI(temperature=0.9, api_key=OPENAI_API_KEY)


first_prompt = ChatPromptTemplate.from_template(
  "Translate this sentence {input} to Spanish Language"
)

chain_one = LLMChain(llm=llm, prompt=first_prompt,
                     output_key="translation" #Give a name to your output
                    )


In [64]:
second_prompt = ChatPromptTemplate.from_template(
    "Summarize the above statement {translation}"
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt,
                     output_key="review" #give a name to this output
                    )

In [65]:
# prompt template 3: translate to english or other language
third_prompt = ChatPromptTemplate.from_template(
    "Translate the summarized review {review} into Hindi"
)
# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key="hindi_translation"
                      )

In [66]:

# prompt template 4: follow up message that take as inputs the two previous prompts' variables
fourth_prompt = ChatPromptTemplate.from_template(
        "Given the original translation: {translation} and its summary: {review}, provide a follow-up message."
)
chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key="fourth"
                     )

In [67]:
# overall_chain: input= Review

review_content = df.Review[0]

# and output= English_Review,summary, followup_message
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["input"],
    output_variables=["translation", "review", "hindi_translation", "fourth"],
    verbose=True
)

In [68]:
review = df.Review[5]
overall_chain({"input": review})

/tmp/ipykernel_6280/4025189161.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  overall_chain({"input": review})




> Entering new SequentialChain chain...

> Finished chain.


{'input': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\nVieux lot ou contrefaçon !?",
 'translation': 'Encuentro que el sabor es mediocre. La espuma no se sostiene, es raro. Compro los mismos en el comercio y el sabor es mucho mejor... ¿Lote antiguo o falsificación!?',
 'review': 'El sabor de los productos mencionados es mediocre y la espuma no se mantiene, lo que lleva a cuestionar si se trata de un lote antiguo o una posible falsificación, ya que al comprar los mismos productos en otro lugar se percibe un sabor mucho mejor.',
 'hindi_translation': 'उल्लिखित उत्पादों का स्वाद मामूली है और फोम नहीं बना रहती है, जिससे संदेह होता है कि क्या यह पुराने बैच का है या क्या यह एक संभावित नकल है, क्योंकि यदि अलग स्थान से उनी उत्पादों को खरीदा जाता है, तो स्वाद मुच्छल बेहतर होता है।',
 'fourth': 'Me gustaría sugerirle que contacte directamente al fabricante para expresar su preocupación sobre la calidad de

**Repeat the above twice for different products or reviews**

## Router Chain

In [69]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts,
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity.

Here is a question:
{input}"""

biology_template = """You are an excellent biologist. \
You have a deep understanding of living organisms, \
from the molecular and cellular level to entire ecosystems. \
You are skilled at observing patterns in nature, analyzing biological data, \
and explaining complex processes like evolution, genetics, physiology, and ecology. \
You can clearly communicate how life functions and adapts, \
and you make connections between different biological concepts \
to answer challenging questions.

Here is a question:
{input}"""

In [70]:
prompt_infos = [
    {
        "name": "physics",
        "description": "Good for answering questions about physics",
        "prompt_template": physics_template
    },
    {
        "name": "math",
        "description": "Good for answering math questions",
        "prompt_template": math_template
    },
    {
        "name": "History",
        "description": "Good for answering history questions",
        "prompt_template": history_template
    },
    {
        "name": "computer science",
        "description": "Good for answering computer science questions",
        "prompt_template": computerscience_template
    },
    {
        "name": "biology",
        "description": "Good for answering biology questions",
        "prompt_template": biology_template
    }
]

In [71]:
from langchain_classic.chains.router import MultiPromptChain
from langchain_classic.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain_core.prompts import PromptTemplate

In [72]:
llm = ChatOpenAI(temperature=0, api_key=OPENAI_API_KEY)

In [73]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain

destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [74]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [75]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

<>:12: SyntaxWarning: invalid escape sequence '\ '
<>:12: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_6280/2160379643.py:12: SyntaxWarning: invalid escape sequence '\ '
  "destination": string \ name of the prompt to use or "DEFAULT"


In [76]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [77]:
chain = MultiPromptChain(router_chain=router_chain,
                         destination_chains=destination_chains,
                         default_chain=default_chain, verbose=True
                        )

/tmp/ipykernel_6280/3038952769.py:1: LangChainDeprecationWarning: Please see migration guide here for recommended implementation: https://python.langchain.com/docs/versions/migrating_chains/multi_prompt_chain/
  chain = MultiPromptChain(router_chain=router_chain,


In [78]:
chain.run("What is black body radiation?")



> Entering new MultiPromptChain chain...
physics: {'input': 'What is black body radiation?'}
> Finished chain.


"Black body radiation refers to the electromagnetic radiation emitted by a perfect black body, which is an idealized physical body that absorbs all incident electromagnetic radiation and emits radiation at all frequencies. The radiation emitted by a black body depends only on its temperature and follows a specific distribution known as Planck's law. This type of radiation is important in understanding concepts such as thermal radiation and the behavior of objects at different temperatures."

In [79]:
chain.run("what is 2 + 2")



> Entering new MultiPromptChain chain...
math: {'input': 'what is 2 + 2'}
> Finished chain.


'The answer to 2 + 2 is 4.'

In [80]:
chain.run("Why does every cell in our body contain DNA?")



> Entering new MultiPromptChain chain...
biology: {'input': 'Why does every cell in our body contain DNA?'}
> Finished chain.


"Every cell in our body contains DNA because DNA is the genetic material that carries the instructions for the development, functioning, and reproduction of all living organisms. DNA contains the information needed to build and maintain an organism, including the proteins that make up our cells and tissues. \n\nHaving DNA in every cell ensures that each cell has the necessary genetic information to carry out its specific functions and to replicate itself accurately during cell division. This ensures that the genetic information is passed on to the next generation of cells, maintaining the integrity and continuity of the organism's genetic code.\n\nAdditionally, DNA serves as a storage system for genetic information that can be accessed and utilized by cells as needed. This allows for the regulation of gene expression, the repair of damaged DNA, and the adaptation to changing environmental conditions.\n\nIn summary, every cell in our body contains DNA because it is essential for the pro

**Repeat the above at least once for different inputs and chains executions - Be creative!**